# Del curso 1 a TensorFlow: el mismo entrenamiento

Entrenar siempre tuvo tres pasos, y no cambian al pasar a redes neuronales:

1. **Definir el modelo** (qué forma tiene la predicción).
2. **Especificar la pérdida y el costo** (cómo se mide el error).
3. **Entrenar** ajustando los parámetros para minimizar el costo.

Lo único que cambia es quién escribe el paso 3: antes un bucle de gradiente descendiente propio; ahora `compile` + `fit`. Este lab empareja paso por paso ambos mundos.

### Recap del curso 1

- **Paso 1 — modelo:** regresión lineal $f_{w,b}(x) = wx + b$, o logística $f_{w,b}(x) = g(wx + b)$.
- **Paso 2 — pérdida y costo:** MSE para regresión, log-loss para clasificación; el costo $J$ promedia la pérdida sobre los $m$ ejemplos (labs 02 y 07).
- **Paso 3 — entrenar:** gradiente descendiente actualizando a mano $w := w - \alpha \, dj\_dw$ y $b := b - \alpha \, dj\_db$ durante muchas iteraciones (labs 03 y 08).

Ese paso 3 es el que más código pedía: calcular gradientes, iterar, vigilar el costo.

### La equivalencia, lado a lado

| Curso 1 | TensorFlow |
|---|---|
| Definir $f_{w,b}$ (recta o sigmoide) | `model = Sequential([Dense(...), ...])` |
| Elegir MSE o log-loss | `model.compile(loss=...)` |
| Gradiente descendiente a mano | `model.compile(optimizer="sgd")` |
| Iteraciones del bucle | `model.fit(..., epochs=N)` |
| Curva de costo impresa | `history.history["loss"]` |

Nada nuevo en matemática: las mismas piezas, ahora empaquetadas en tres llamadas.

### Paso 1 en TensorFlow: definir el modelo

Igual que en el curso 1 se elegía recta o sigmoide, aquí se elige la arquitectura: cuántas capas, cuántas neuronas y qué activación. El dataset es el de tumores en 2D (tamaño y densidad); la red tiene una capa oculta y una salida con sigmoide para dar probabilidad.

(El detalle de `relu` y `sigmoid` como funciones de activación es el tema del siguiente lab; por ahora basta saber que moldean la salida de cada capa.)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, layers

tf.random.set_seed(7)
np.random.seed(7)

# Dataset de tumores: tamaño (cm) y densidad, con etiqueta benigno/maligno
m = 120
tam_ben = np.random.normal(3.0, 1.0, m // 2)
den_ben = np.random.normal(12.0, 1.5, m // 2)
tam_mal = np.random.normal(6.5, 1.0, m // 2)
den_mal = np.random.normal(17.5, 1.5, m // 2)
X = np.column_stack([np.concatenate([tam_ben, tam_mal]),
                     np.concatenate([den_ben, den_mal])]).astype(np.float32)
y = np.array([0] * (m // 2) + [1] * (m // 2), dtype=np.float32)
print(f"X: {X.shape} | y: {y.shape} (0 benigno, 1 maligno)")

# Paso 1: definir el modelo
model = Sequential([
    layers.Dense(units=8, activation="relu"),
    layers.Dense(units=1, activation="sigmoid"),
])
model.build(input_shape=(None, 2))
model.summary()

### Paso 2 en TensorFlow: `compile` = pérdida + optimizador

`compile` declara las dos decisiones del paso 2:

- **`loss="binary_crossentropy"`**: es **exactamente el log-loss del Lab 07** con otro nombre. Binary porque hay dos clases; crossentropy porque mide la diferencia entre la distribución real y la predicha. En TensorFlow también aparece como `tf.keras.losses.BinaryCrossentropy()`.
- **`optimizer="sgd"`**: stochastic gradient descent, el gradiente descendiente del Lab 08 (por lotes). Aquí solo se elige; el ajuste ocurre en el paso 3.
- **`metrics=["accuracy"]`**: solo para observar el desempeño, no participa del entrenamiento.

In [ ]:
# Paso 2: especificar pérdida (log-loss) y optimizador (gradiente descendiente)
model.compile(
    optimizer="sgd",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
print(f"loss configurada: {model.loss}")
print(f"optimizador: {model.optimizer.__class__.__name__}")
print("binary_crossentropy = el log-loss del Lab 07, con nombre nuevo")

### Paso 3 en TensorFlow: `fit` y las épocas

`model.fit(X, y, epochs=N)` hace el papel del bucle de gradiente descendiente del curso 1, con una diferencia de vocabulario:

- Una **época** es una pasada completa por todos los datos de entrenamiento.
- Dentro de cada época, los datos se parten en **lotes** (batch); cada lote produce una actualización de pesos. Con 120 ejemplos y `batch_size=16` hay 8 actualizaciones por época; con 100 épocas, 800 actualizaciones en total.
- `history.history["loss"]` guarda el costo al final de cada época: la misma curva de convergencia del Lab 08, ahora gratis.

In [ ]:
import matplotlib.pyplot as plt

# Estilo para las gráficas
plt.style.use('ggplot')

# Paso 3: entrenar (cada época = una pasada completa por los datos)
batch_size = 16
epochs = 100
history = model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
import math
actualizaciones = math.ceil(m / batch_size) * epochs
perdida = history.history["loss"]
print(f"Épocas: {epochs} | lotes por época: {math.ceil(m / batch_size)} | actualizaciones de pesos: {actualizaciones}")
print(f"Costo (loss) inicial: {perdida[0]:.3f} | final: {perdida[-1]:.3f}")
print(f"¿Bajó el costo? {perdida[-1] < perdida[0]}")

plt.plot(perdida, c='purple')
plt.title("El costo por época: la misma convergencia del Lab 08")
plt.xlabel("Época")
plt.ylabel("Binary crossentropy")
plt.show()

### Cierre

Los tres pasos no cambiaron, solo su empaquetado: definir el modelo sigue siendo elegir la forma de $f$, especificar el costo sigue siendo elegir entre MSE y log-loss (aquí `binary_crossentropy`), y entrenar sigue siendo gradiente descendiente (aquí `sgd` dentro de `fit`). Las épocas son las iteraciones del bucle, contadas por pasadas completas a los datos.

Lo que sigue: las funciones de activación (`relu`, `sigmoid`, `tanh`) que ya aparecieron en el modelo y que merecen su propio lab, y después la evaluación y el ajuste fino del entrenamiento.